In [ ]:
import os
import re

import requests
from bs4 import BeautifulSoup

from src.constants import CROSSHARE_PUZ_DIR


In [2]:
def extract_crossword_id(text):
    match = re.search(r"(?<=/crosswords/)(.*)(?=/)", text)

    if match:
        return match.group(1)
    return None

In [ ]:
def download_crossword(crossword_id, rating, index):
    filename = f"{CROSSHARE_PUZ_DIR}/{rating}_{index}_{crossword_id}.puz"

    if os.path.exists(filename):
        print(f"File already exists: {os.path.abspath(filename)}")
        return

    url = f"https://crosshare.org/api/puz/{crossword_id}"

    response = requests.get(url)

    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)

        print(f"Success! File saved at: {os.path.abspath(filename)}")
    else:
        print(f"Download failed. Status: {response.status_code}")

In [4]:
def scrape_page(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    links = soup.find_all("a")

    crossword_ids = set()
    for link in links:
        href = link.get("href")
        crossword_id = extract_crossword_id(href)

        if crossword_id:
            crossword_ids.add(crossword_id)

    return crossword_ids

In [5]:
def scrape_crosswords_ids():
    crossword_ids = set()

    page = 0
    while True:
        url = f"https://crosshare.org/tags/rating-{rating}/page/{page}"
        print(f"Scraping page: {url}")

        new_crossword_ids = scrape_page(url)

        if not new_crossword_ids:
            print("No more crosswords found. Ending scrape.")
            break

        crossword_ids.update(new_crossword_ids)
        page += 1

    return crossword_ids

In [6]:
os.makedirs("crosshare_puz_files", exist_ok=True)
rating = "0-1200"

crossword_ids = scrape_crosswords_ids()

for index, crossword_id in enumerate(crossword_ids):
    download_crossword(crossword_id, rating, index)

Scraping page: https://crosshare.org/tags/rating-0-1200/page/0
Scraping page: https://crosshare.org/tags/rating-0-1200/page/1
Scraping page: https://crosshare.org/tags/rating-0-1200/page/2
Scraping page: https://crosshare.org/tags/rating-0-1200/page/3
Scraping page: https://crosshare.org/tags/rating-0-1200/page/4
Scraping page: https://crosshare.org/tags/rating-0-1200/page/5
Scraping page: https://crosshare.org/tags/rating-0-1200/page/6
Scraping page: https://crosshare.org/tags/rating-0-1200/page/7
Scraping page: https://crosshare.org/tags/rating-0-1200/page/8
Scraping page: https://crosshare.org/tags/rating-0-1200/page/9
Scraping page: https://crosshare.org/tags/rating-0-1200/page/10
No more crosswords found. Ending scrape.
Success! File saved at: /home/patel/code/crossword-solver/crosshare_puz_files/0-1200_0_BMPHMYVG6af9cIv0L3He.puz
Success! File saved at: /home/patel/code/crossword-solver/crosshare_puz_files/0-1200_1_xpPyQ3Vz4a9NMCC5UsBm.puz
Success! File saved at: /home/patel/code/